# Clearcut forest vs. agriculture in Florida — LANDFIRE EVT change

**Method 2 of 2.** A pixel that LANDFIRE EVT shows going from a **forest type** to one of the three suspect classes (pasture/hay, ruderal grassland, floodplain shrubland) is evidence of a **clearcut**, not genuine farmland. This notebook builds that year-over-year change detector and compares it against Method 1 (embeddings) and the independent LCMS tree-removal signal.

**Data-availability note.** GEE hosts only a single LANDFIRE EVT vintage (`LANDFIRE/Vegetation/EVT/v1_4_0`, ~2016, pre-Remap codes). There is no annual EVT series on GEE, so we pair GEE EVT 2016 (*before*) with the local LF2022 EVT tif (*after*). Because the two vintages use different code schemes, the *before* state is reduced to a stable **forest-vs-not** call from the EVT class *name*, and the *after* state uses the LF2022 class code / lifeform.

Design: `docs/superpowers/specs/2026-07-01-clearcut-vs-agriculture-embeddings-design.md`

## 1. Setup

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
_candidates = [cwd / "notebooks", cwd, *[p / "notebooks" for p in cwd.parents], *cwd.parents]
nb_dir = next(p for p in _candidates if (p / "clearcut_ag_common.py").exists())
sys.path.insert(0, str(nb_dir))

import clearcut_ag_common as cac

REPO_ROOT = cac.find_repo_root(nb_dir)
ee = cac.init_ee()
florida, florida_gdf = cac.load_florida(REPO_ROOT)
print("Repo root:", REPO_ROOT)
print("Florida extent:", florida_gdf[["name", "fips"]].to_dict("records"))

Repo root: /home/chazm/projects/artemis-model
Florida extent: [{'name': 'Florida', 'fips': '12'}]


## 2. Load the shared sample points
Reuse the identical points/labels the Embeddings notebook wrote (so the two methods are compared on the same locations). If that file is absent, regenerate the table (without the embedding classifier prediction).

In [2]:
import pandas as pd

OUT_DIR = REPO_ROOT / "data" / "interim" / "clearcut_ag"
csv_path = OUT_DIR / "embeddings_samples.csv"
if csv_path.exists():
    samples = pd.read_csv(csv_path)
    print("Loaded shared sample table:", len(samples), "rows from", csv_path)
else:
    print("Shared table not found — regenerating points (run the Embeddings notebook "
          "first to reuse identical points and enable the cross-method comparison).")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    samples = cac.build_sample_table(florida, REPO_ROOT)
    samples = samples.dropna(subset=list(cac.EMBEDDING_BANDS)).reset_index(drop=True)
print(samples["label"].value_counts())

Loaded shared sample table: 1700 rows from /home/chazm/projects/artemis-model/data/interim/clearcut_ag/embeddings_samples.csv
label
other          1129
clearcut        304
confused        182
agriculture      85
Name: count, dtype: int64


## 3. EVT year-over-year change flags
`evt_change_strict` = forest in 2016 → one of the three confused classes in 2022. `evt_change_broad` = forest in 2016 → any ag/grass/shrub lifeform in 2022.

In [3]:
evt2022_lookup = cac.load_evt2022_lookup(cac.evt2022_csv_path(REPO_ROOT))

samples["evt_change_strict"] = samples.apply(
    lambda r: cac.evt_change_clearcut(bool(r["is_forest_2016"]), r["evt2022"]), axis=1
)
samples["evt_change_broad"] = samples.apply(
    lambda r: cac.evt_change_clearcut_broad(bool(r["is_forest_2016"]), r["evt2022"], evt2022_lookup),
    axis=1,
)
print("Forest(2016) -> confused class (strict):", int(samples.evt_change_strict.sum()))
print("Forest(2016) -> ag/grass/shrub (broad): ", int(samples.evt_change_broad.sum()))

Forest(2016) -> confused class (strict): 24
Forest(2016) -> ag/grass/shrub (broad):  43


## 4. Change-flag rate by label group

In [4]:
rate = (
    samples.groupby("label")
    .agg(
        n=("evt_change_strict", "size"),
        strict_rate=("evt_change_strict", "mean"),
        broad_rate=("evt_change_broad", "mean"),
        lcms_treeremoval_rate=("is_clearcut", "mean"),
    )
    .round(3)
)
print(rate)

                n  strict_rate  broad_rate  lcms_treeremoval_rate
label                                                            
agriculture    85        0.000       0.000                  0.000
clearcut      304        0.000       0.010                  1.000
confused      182        0.132       0.132                  0.011
other        1129        0.000       0.014                  0.000


## 5. Agreement with the LCMS tree-removal signal
Treat LCMS `Tree Removal` (pre-forest → removal) as an independent clearcut reference and score each EVT-change flag as a detector against it.

In [5]:
ref = samples["is_clearcut"].astype(bool)
for flag in ["evt_change_strict", "evt_change_broad"]:
    pred = samples[flag].astype(bool)
    tp = int((pred & ref).sum()); fp = int((pred & ~ref).sum()); fn = int((~pred & ref).sum())
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    rec = tp / (tp + fn) if (tp + fn) else float("nan")
    print(f"{flag}: precision={prec:.3f} recall={rec:.3f} (tp={tp} fp={fp} fn={fn})")

evt_change_strict: precision=0.042 recall=0.003 (tp=1 fp=23 fn=305)
evt_change_broad: precision=0.093 recall=0.013 (tp=4 fp=39 fn=302)


## 6. Cross-method comparison
For the three confused EVT classes, what share does each method flag as clearcut? `embed_clearcut` needs Method 1's `clearcut_prob` (present only when the shared table came from the Embeddings notebook).

In [6]:
if "clearcut_prob" in samples.columns:
    samples["embed_pred_clearcut"] = samples["clearcut_prob"] > 0.5
    conf = samples[samples.label == "confused"]
    comp = (
        conf.groupby("confused_name")
        .agg(
            n=("evt_change_strict", "size"),
            embed_clearcut=("embed_pred_clearcut", "mean"),
            evt_change_strict=("evt_change_strict", "mean"),
            lcms_treeremoval=("is_clearcut", "mean"),
        )
        .round(3)
    )
    print("Share of each confused EVT class flagged as clearcut, by method:")
    print(comp)
else:
    print("No embedding predictions in the table; run the Embeddings notebook to enable this comparison.")

Share of each confused EVT class flagged as clearcut, by method:
                    n  embed_clearcut  evt_change_strict  lcms_treeremoval
confused_name                                                             
floodplain_shrub    1           1.000              1.000             0.000
pasture_hay       144           0.340              0.062             0.014
ruderal_grass      37           0.838              0.378             0.000


## 7. Persist the enriched table

In [7]:
out_path = OUT_DIR / "evt_change_samples.csv"
samples.to_csv(out_path, index=False)
print("Wrote", out_path)

Wrote /home/chazm/projects/artemis-model/data/interim/clearcut_ag/evt_change_samples.csv


## Interpretation notes
- **Precision vs LCMS** tells you how often an EVT forest→ag/grass/shrub transition is a real clearcut. High precision means the EVT-change flag is a trustworthy clearcut detector; low recall is expected because EVT vintages are years apart and coarse.
- The **strict** flag targets exactly the three suspect classes; the **broad** flag trades precision for recall.
- Caveat: the 2016↔2022 comparison bridges two EVT code schemes at the forest-vs-not level only. A clean same-scheme diff (LF2016 vs LF2022) would need a second local EVT download and is the natural follow-up.